# 83 · Titanic — Tuning de hiperparámetros y registro del champion (Experimento E4)

Cuarta parte. Ajustar los hiperparámetros de un *Random Forest* con validación cruzada, comparas todos
los experimentos con `mlflow.search_runs`, eliges el **champion** y lo registras en Unity Catalog con un
**alias**.

### Objetivos de aprendizaje
- Construir una malla de hiperparámetros y ejecutar `CrossValidator` (3 folds).
- Registrar cada combinación como *nested run* de MLflow.
- Extraer la importancia de variables del mejor *Random Forest*.
- Comparar todos los *runs* con `mlflow.search_runs` y persistir el resumen.
- Registrar el modelo ganador en Unity Catalog y asignarle el alias `champion`.


## 1. Configuración

In [0]:
import os

import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

CATALOGO = "big_data_ii_2025"
ESQUEMA = "spark_examples"
TABLA_SILVER_TRAIN = f"{CATALOGO}.{ESQUEMA}.titanic_silver_train"
TABLA_RESULTADOS = f"{CATALOGO}.{ESQUEMA}.titanic_resultados_experimentos"
NOMBRE_MODELO_UC = f"{CATALOGO}.{ESQUEMA}.titanic_modelo_supervivencia"
RUTA_EXPERIMENTO = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic_mlflow"
RUTA_DF_TEMPORAL = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic_mlflow_tmp"
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.titanic_mlflow_tmp
""")
# MLflow usará este directorio al guardar modelos Spark ML
os.environ["MLFLOW_DFS_TMP"] = RUTA_EXPERIMENTO
os.environ["SPARKML_TEMP_DFS_PATH"]=RUTA_DF_TEMPORAL

mlflow.set_registry_uri("databricks-uc")
usuario = spark.sql("SELECT current_user()").first()[0]
RUTA_EXPERIMENTO_USUARIO = f"/Users/{usuario}/titanic_mlflow"
mlflow.set_experiment(RUTA_EXPERIMENTO_USUARIO)
print("Experimento:", RUTA_EXPERIMENTO)

## 2. Datos y preprocesamiento

Reutilizamos la capa silver y el mismo *split* 80/20 con semilla 42, para que las métricas de
validación sean comparables con E2 y E3.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
)

silver = spark.table(TABLA_SILVER_TRAIN)
silver_ent, silver_val = silver.randomSplit([0.8, 0.2], seed=42)

CATEGORICAS = ["Sex", "Title", "EmbarkedFill", "Deck"]
NUMERICAS = ["Pclass", "AgeImputed", "FareImputed", "FamilySize", "IsAlone"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in CATEGORICAS
]
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in CATEGORICAS],
    outputCols=[f"{c}_ohe" for c in CATEGORICAS],
)
assembler = VectorAssembler(
    inputCols=NUMERICAS + [f"{c}_ohe" for c in CATEGORICAS],
    outputCol="features",
    handleInvalid="keep",
)

rf = RandomForestClassifier(featuresCol="features", labelCol="Survived", seed=42)
pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])


def evaluar(pred_df):
    acc = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="accuracy").evaluate(pred_df)
    f1 = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="f1").evaluate(pred_df)
    auc = BinaryClassificationEvaluator(
        labelCol="Survived", rawPredictionCol="probability", metricName="areaUnderROC").evaluate(pred_df)
    return acc, f1, auc

## 3. Malla de hiperparámetros

Mantenemos la malla pequeña para no agotar la cuota de *compute* de Free Edition:

| Hiperparámetro | Valores |
|---|---|
| `numTrees` | 50, 100 |
| `maxDepth` | 4, 6, 8 |
| `minInstancesPerNode` | 1, 5 |

> **Antes de ejecutar:** con 3 folds y esta malla, ¿cuántos entrenamientos (*fits*) hará el
> `CrossValidator` en total? Multiplica las combinaciones por los folds antes de correr la celda.

In [0]:
# malla = (
#     ParamGridBuilder()
#     .addGrid(rf.numTrees, [50, 100])
#     .addGrid(rf.maxDepth, [4, 6, 8])
#     .addGrid(rf.minInstancesPerNode, [1, 5])
#     .build()
# )

# NOTA: Reduccion de tamaño de la malla para pruebas
# Para evitar errores de cache de ML lleno en serverless free tier
malla = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [25, 50])
    .addGrid(rf.maxDepth, [3])
    .addGrid(rf.minInstancesPerNode, [1])
    .build()
)

n_combinaciones = len(malla)
# n_folds = 3
n_folds = 1
n_fits = n_combinaciones * n_folds
print(f"Combinaciones: {n_combinaciones}")
print(f"Folds: {n_folds}")
print(f"Fits totales: {n_fits}")

# Verificación de la predicción: 2*3*2 = 12 combinaciones, 12*3 = 36 fits.
# assert n_combinaciones == 12, f"Se esperaban 12 combinaciones, hay {n_combinaciones}"
# assert n_fits == 36, f"Se esperaban 36 fits, hay {n_fits}"
# print("Correcto: 12 combinaciones × 3 folds = 36 fits.")

## 4. Validación cruzada con *nested runs*

Ejecutamos el ~CrossValidator~ `TrainValidationSplit` dentro de un *run* padre. 

In [0]:
import gc
from pyspark.ml.tuning import TrainValidationSplit

# Deshabilitar autologging porque no es compatible con Spark Connect
# en Databricks Free Edition con Serverless.
mlflow.autolog(disable=True)
mlflow.pyspark.ml.autolog(disable=True)

evaluador_cv = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="accuracy",
)

# NOTA: Usar TrainValidationSplit en lugar de CrossValidator para evitar
# el límite de cache de ML en serverless free tier.
# TrainValidationSplit hace un solo split train/val (no folds),
# entrenando solo 2 modelos en total en lugar de 4.
tv = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=malla,
    evaluator=evaluador_cv,
    trainRatio=0.8,
    seed=42,
    parallelism=1,
)

# Registro manual de MLflow.
with mlflow.start_run(run_name="E4_rf_cv_tuning") as run_padre:

    mlflow.set_tags({
        "experimento": "E4",
        "modelo": "RandomForest_CV",
        "notebook": "83",
    })

    # Entrenar todas las combinaciones de hiperparámetros.
    modelo_tv = tv.fit(silver_ent)

    # Extraer métricas y mejor modelo inmediatamente.
    validation_metrics = [float(v) for v in modelo_tv.validationMetrics]
    mejor_modelo = modelo_tv.bestModel
    
    # Liberar el TrainValidationSplitModel inmediatamente.
    del modelo_tv
    gc.collect()

    # Guardar los resultados de validación de todas las combinaciones.
    resultados_cv = {
        "metric_name": "accuracy",
        "validation_metrics": validation_metrics,
    }

    # Evaluar el mejor modelo con el conjunto de validación.
    pred_val = mejor_modelo.transform(silver_val)
    acc_e4, f1_e4, auc_e4 = evaluar(pred_val)
    
    # Generar una salida representativa.
    salida_spark = (
        pred_val.select("prediction")
    )

    # Liberar predicciones inmediatamente.
    del pred_val
    gc.collect()

    # El Random Forest es la última etapa del pipeline.
    rf_ajustado = mejor_modelo.stages[-1]

    mejores_params = {
        "numTrees": int(rf_ajustado.getNumTrees),
        "maxDepth": int(
            rf_ajustado.getOrDefault(
                rf_ajustado.maxDepth
            )
        ),
        "minInstancesPerNode": int(
            rf_ajustado.getOrDefault(
                rf_ajustado.minInstancesPerNode
            )
        ),
    }

    # Registrar configuración general del experimento.
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "tuning_method": "TrainValidationSplit",
        "trainRatio": 0.8,
        "seed": 42,
        "parallelism": 1,
        "num_param_combinations": len(malla),
    })

    # Registrar los hiperparámetros seleccionados.
    mlflow.log_params({
        f"mejor_{nombre}": valor
        for nombre, valor in mejores_params.items()
    })

    # Registrar métricas del mejor modelo.
    mlflow.log_metrics({
        "accuracy": float(acc_e4),
        "f1": float(f1_e4),
        "auc_roc": float(auc_e4),
    })



    mlflow.log_dict(
        resultados_cv,
        "cross_validation/cv_results.json",
    )

    # Crear un ejemplo de entrada para inferir la firma del modelo.
    ejemplo_entrada = silver_ent.limit(5).toPandas().drop(columns=["Survived"])

    # Registrar únicamente el mejor modelo.
    mlflow.spark.log_model(
        spark_model=mejor_modelo,
        artifact_path="model",
        dfs_tmpdir=RUTA_EXPERIMENTO,
        input_example=ejemplo_entrada,
        signature=mlflow.models.infer_signature(
            model_input=ejemplo_entrada, 
            model_output=salida_spark)
    )
        # signature=mlflow.models.infer_signature(ejemplo_entrada, mejor_modelo.transform(silver_ent.limit(5)).toPandas())

    run_id_e4 = run_padre.info.run_id

print("--- E4: mejor modelo de validación cruzada ---")
print("Mejores hiperparámetros:", mejores_params)
print(
    f"accuracy={acc_e4:.4f}  "
    f"f1={f1_e4:.4f}  "
    f"auc={auc_e4:.4f}"
)
print("run_id padre:", run_id_e4)

## 5. Importancia de variables del champion

El *Random Forest* expone `featureImportances`. Lo cruzamos con los nombres del `VectorAssembler` para
leerlo. Guarda el gráfico como artefacto del *run* de E4.

In [0]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Nombres de features desde los metadatos del vector ensamblado.
pred_metadatos = mejor_modelo.transform(silver_ent.limit(1))
attrs = pred_metadatos.schema["features"].metadata["ml_attr"]["attrs"]
indice_a_nombre = {}
for grupo in attrs.values():
    for a in grupo:
        indice_a_nombre[a["idx"]] = a["name"]

importancias = rf_ajustado.featureImportances
pares = sorted(
    [(indice_a_nombre.get(i, f"f{i}"), float(importancias[i])) for i in range(len(importancias))],
    key=lambda x: x[1], reverse=True,
)[:12]
nombres = [p[0] for p in pares][::-1]
valores = [p[1] for p in pares][::-1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(nombres, valores, color="#FF871E")
ax.set_title("E4 · Importancia de variables (top 12)")
ax.set_xlabel("Importancia")
fig.tight_layout()
ruta_fi = "/tmp/feature_importances_e4.png"
fig.savefig(ruta_fi, dpi=100)
plt.close(fig)

# Adjuntamos el gráfico al run padre de E4.
with mlflow.start_run(run_id=run_id_e4):
    mlflow.log_artifact(ruta_fi, artifact_path="graficos")

print("Top variables por importancia:")
for nombre, val in pares:
    print(f"  {nombre:24s} {val:.4f}")

## 6. Compara todos los experimentos y persiste el resumen

Con `mlflow.search_runs` traemos todos los *runs* del experimento ordenados por *accuracy* y guardamos
el resumen en una tabla Delta.

In [0]:
exp = mlflow.get_experiment_by_name(RUTA_EXPERIMENTO_USUARIO)
runs_pdf = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.accuracy DESC"],
)

columnas_interes = [
    "run_id", "tags.mlflow.runName", "tags.experimento", "tags.modelo",
    "metrics.accuracy", "metrics.f1", "metrics.auc_roc",
]
columnas_presentes = [c for c in columnas_interes if c in runs_pdf.columns]
resumen_pdf = runs_pdf[columnas_presentes].copy()

# Nos quedamos solo con runs que tengan accuracy (los runs con nombre E1..E4).
resumen_pdf = resumen_pdf[resumen_pdf["metrics.accuracy"].notna()]
resumen_pdf = resumen_pdf.rename(columns={
    "tags.mlflow.runName": "run_name",
    "tags.experimento": "experimento",
    "tags.modelo": "modelo",
    "metrics.accuracy": "accuracy",
    "metrics.f1": "f1",
    "metrics.auc_roc": "auc_roc",
})

df_resumen = spark.createDataFrame(resumen_pdf)
df_resumen.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_RESULTADOS)
print(f"Resumen de experimentos guardado en {TABLA_RESULTADOS}")
df_resumen.orderBy(F.desc("accuracy")).display()

## 7. Selecciona el champion y regístralo en Unity Catalog

El *run* con mayor *accuracy* de validación es nuestro **champion**. Lo registramos como versión del
modelo `titanic_modelo_supervivencia` en Unity Catalog y le asignamos el **alias** `champion`.

Nota: usamos **aliases**, no *stages*. Los *stages* (`Staging`/`Production`) están descontinuados en el
Model Registry de Unity Catalog.

In [0]:
fila_champion = resumen_pdf.sort_values("accuracy", ascending=False).iloc[0]
run_id_champion = fila_champion["run_id"]
acc_champion = float(fila_champion["accuracy"])
print(f"Champion → run_id={run_id_champion}  accuracy={acc_champion:.4f}  "
      f"({fila_champion.get('run_name', 'sin nombre')})")

# Detectamos el nombre real de la carpeta del modelo dentro del run.
# El log manual (E1) y el autolog de SparkML (E2-E4) usan 'model' o 'sparkml';
# lo resolvemos en lugar de asumirlo, para que funcione sea cual sea el champion.
client = MlflowClient()
artefactos = [a.path for a in client.list_artifacts(run_id_champion)]
print("Artefactos del run champion:", artefactos)

carpeta_modelo = None
for candidato in ["model", "sparkml", "best_model"]:
    if candidato in artefactos:
        carpeta_modelo = candidato
        break
if carpeta_modelo is None:
    # Fallback: primer artefacto que sea un directorio de modelo (contiene MLmodel).
    for a in artefactos:
        sub = [s.path.split("/")[-1] for s in client.list_artifacts(run_id_champion, a)]
        if "MLmodel" in sub:
            carpeta_modelo = a
            break
assert carpeta_modelo is not None, (
    f"No se encontró la carpeta del modelo en el run {run_id_champion}. "
    f"Artefactos disponibles: {artefactos}"
)
uri_modelo = f"runs:/{run_id_champion}/{carpeta_modelo}"
print("URI del modelo champion:", uri_modelo)

version = mlflow.register_model(model_uri=uri_modelo, name=NOMBRE_MODELO_UC)
print(f"Registrado {NOMBRE_MODELO_UC} versión {version.version}")

client.set_registered_model_alias(
    name=NOMBRE_MODELO_UC, alias="champion", version=version.version
)
print(f"Alias 'champion' → versión {version.version}")

## 8. Explora el modelo registrado en la interfaz

1. Abre **Catalog → `big_data_ii_2025` → `spark_examples` → sección Models → `titanic_modelo_supervivencia`**.
2. Revisa las versiones, el alias **`@champion`**, la trazabilidad (*lineage*) al *run* de origen y la
   firma (esquema de entrada y salida).
3. Para mover el alias a otra versión: menú ⋮ de la versión → **Set alias**. Así se promueve un nuevo
   modelo sin tocar el código de inferencia, porque este apunta al alias, no a un número de versión.

## Qué aprendiste en esta parte
- Construiste una malla de hiperparámetros y ejecutaste `CrossValidator` (36 *fits*).
- Registraste el tuning con *nested runs* y extrajiste la importancia de variables.
- Comparaste todos los experimentos con `mlflow.search_runs` y guardaste el resumen.
- Registraste el **champion** en Unity Catalog con la firma y le asignaste el alias `champion`.

**Siguiente parte → `84_titanic_inferencia_submission`:** aplicarás el champion al `test.csv`, generarás
la tabla de predicciones y exportarás el CSV para hacer *submit* en Kaggle.